In [1]:
import pandas as pd

# -------------------------------
# 1) Load ground truth (drop TotalCells, collect sorted cols)
# -------------------------------
gt_files = {
    "GT_BDa_Outer": "COO-Decon-Pseudobulks/TSP-BDa_Outer_Random_v2C_All-Proportions.txt",
    "GT_BDa_Inner": "COO-Decon-Pseudobulks/TSP-BDa_Inner_Random_v2C_All-Proportions.txt",
    "GT_HBA":       "COO-Decon-Pseudobulks/TSP-HBA_Inner_Random_v2C_All-Proportions.txt",
}

gt_data = {}
for name, path in gt_files.items():
    df = pd.read_csv(path, sep="\t", index_col=0)
    if "TotalCells" in df.columns:
        df = df.drop(columns=["TotalCells"])
    gt_data[name] = sorted(df.columns.tolist())

# -------------------------------
# 2) Load BayesPrism files
# -------------------------------
bp_files = {
    "BP_BDa_Inner_top10": "BayesPrism-Deconvolutions/TSP-BDa_Inner_100each_seed42_filtered_Random_v2C_top_10_percent_removed_BayesPrism.txt",
    "BP_BDa_Inner_top20": "BayesPrism-Deconvolutions/TSP-BDa_Inner_100each_seed42_filtered_Random_v2C_top_20_percent_removed_BayesPrism.txt",
    "BP_BDa_Inner_top30": "BayesPrism-Deconvolutions/TSP-BDa_Inner_100each_seed42_filtered_Random_v2C_top_30_percent_removed_BayesPrism.txt",
    "BP_BDa_Inner_top40": "BayesPrism-Deconvolutions/TSP-BDa_Inner_100each_seed42_filtered_Random_v2C_top_40_percent_removed_BayesPrism.txt"
}

# -------------------------------
# 3) Build rename dictionaries from previous comparison file
# -------------------------------
comparison_df = pd.read_csv(
    "COO-Decon-Pseudobulks/Column_Comparison_GT_vs_BayesPrism_Final.txt", 
    sep="\t"
)

rename_map_bda_outer = dict(
    zip(comparison_df["BP_BDa_Outer"].dropna(), comparison_df["GT_BDa_Outer"].dropna())
)
rename_map_bda_inner = dict(
    zip(comparison_df["BP_BDa_Inner"].dropna(), comparison_df["GT_BDa_Inner"].dropna())
)
rename_map_hba = dict(
    zip(comparison_df["BP_HBA"].dropna(), comparison_df["GT_HBA"].dropna())
)

# -------------------------------
# 4) Apply manual fixes for known mismatches
# -------------------------------
rename_map_hba.update({
    "basal.cell.of.prostate.epithelium": "basal cell of prostate epithelium",
    "basal.cell.medullary.thymic.epithelial.cell": "basal cell/medullary thymic epithelial cell",
})
rename_map_bda_outer.update({
    "endothelial.cell.endothelial.cell.of.lymphatic.vessel.vein.endothelial.cell":
        "endothelial cell/endothelial cell of lymphatic vessel/vein endothelial cell",
    "endothelial.cell.of.arteriole.endothelial.cell.of.venule":
        "endothelial cell of arteriole/endothelial cell of venule",
})
rename_map_bda_inner.update({
    "endothelial.cell.endothelial.cell.of.lymphatic.vessel.vein.endothelial.cell":
        "endothelial cell/endothelial cell of lymphatic vessel/vein endothelial cell",
    "endothelial.cell.of.arteriole.endothelial.cell.of.venule":
        "endothelial cell of arteriole/endothelial cell of venule",
})

rename_maps = {
    "BP_BDa_Inner_top10": rename_map_bda_inner,
    "BP_BDa_Inner_top20": rename_map_bda_inner,
    "BP_BDa_Inner_top30": rename_map_bda_inner,
    "BP_BDa_Inner_top40": rename_map_bda_inner
}

# -------------------------------
# 5) Load BP again, apply rename maps, collect sorted cols
# -------------------------------
bp_data = {}
bp_frames = {}
for name, path in bp_files.items():
    df = pd.read_csv(path, sep="\t", index_col=0)
    df = df.rename(columns=rename_maps[name])
    bp_frames[name] = df
    bp_data[name] = sorted(df.columns.tolist())

# -------------------------------
# 6) Build side-by-side comparison (sorted)
# -------------------------------
cols = {
    "GT_BDa_Inner": gt_data["GT_BDa_Inner"],
    "BP_BDa_Inner_top10": bp_data["BP_BDa_Inner_top10"],
    "BP_BDa_Inner_top20": bp_data["BP_BDa_Inner_top20"],
    "BP_BDa_Inner_top30": bp_data["BP_BDa_Inner_top30"],
    "BP_BDa_Inner_top40": bp_data["BP_BDa_Inner_top40"]
}

max_len = max(len(v) for v in cols.values())
for k in cols:
    cols[k] = cols[k] + [""] * (max_len - len(cols[k]))

final_comparison = pd.DataFrame(cols)

# -------------------------------
# 7) Save updated comparison + check matches
# -------------------------------
final_comparison.to_csv("Column_Comparison_GT_vs_BayesPrism_AfterRenaming.txt", sep="\t", index=False)

for pair in [
    ("GT_BDa_Inner", "BP_BDa_Inner_top10"),
    ("GT_BDa_Inner", "BP_BDa_Inner_top20"),
    ("GT_BDa_Inner", "BP_BDa_Inner_top30"),
    ("GT_BDa_Inner", "BP_BDa_Inner_top40")    
]:
    gt_set = set(gt_data[pair[0]])
    bp_set = set(bp_data[pair[1]])
    common = gt_set & bp_set
    print(f"{pair[1]}: {len(common)} / {len(gt_set)} columns match GT")

# -------------------------------
# 8) Write out renamed BP files (with _renamed.txt suffix)
# -------------------------------
for name, path in bp_files.items():
    out_path = path.replace(".txt", "_renamed.txt")
    bp_frames[name].to_csv(out_path, sep="\t")
    print(f"Saved renamed file: {out_path}")


BP_BDa_Inner_top10: 80 / 80 columns match GT
BP_BDa_Inner_top20: 80 / 80 columns match GT
BP_BDa_Inner_top30: 80 / 80 columns match GT
BP_BDa_Inner_top40: 80 / 80 columns match GT
Saved renamed file: BayesPrism-Deconvolutions/TSP-BDa_Inner_100each_seed42_filtered_Random_v2C_top_10_percent_removed_BayesPrism_renamed.txt
Saved renamed file: BayesPrism-Deconvolutions/TSP-BDa_Inner_100each_seed42_filtered_Random_v2C_top_20_percent_removed_BayesPrism_renamed.txt
Saved renamed file: BayesPrism-Deconvolutions/TSP-BDa_Inner_100each_seed42_filtered_Random_v2C_top_30_percent_removed_BayesPrism_renamed.txt
Saved renamed file: BayesPrism-Deconvolutions/TSP-BDa_Inner_100each_seed42_filtered_Random_v2C_top_40_percent_removed_BayesPrism_renamed.txt
